Set B: Context Length Robustness Evaluation

Research Question:
RQ3: How does quantization impact performance across varying context lengths?

Evaluation Coverage:
- Context length stress test: 512, 1024, 2048, 4096 tokens
- Answer position sensitivity: start, middle, end
- Degradation curves and position effects
- Cross-configuration comparison

Setup

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple
import re
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Imports complete")

In [ ]:
INPUT_DIR = Path('/kaggle/input/generation-sets')
OUTPUT_DIR = Path('/kaggle/working/set_b_evaluation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 1000
CONFIDENCE_LEVEL = 0.95
ALPHA = 0.05

CONTEXT_LENGTHS = [512, 1024, 2048, 4096]
ANSWER_POSITIONS = ['start', 'middle', 'end']

np.random.seed(RANDOM_SEED)

print(f"Input: {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

Normalization and Metrics

In [ ]:
def normalize_answer(text: str) -> str:
    """Normalize text following SQuAD evaluation protocol"""
    import unicodedata
    
    if not text:
        return ""
    
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = ' '.join(text.split())
    
    return text.strip()

def extract_conservative(text: str, ground_truth: str = None, max_ratio: float = 3.0) -> str:
    """Conservative extraction strategy"""
    if not text:
        return ""
    
    text = text.strip()
    
    prefixes = ['Answer:', 'answer:', 'A:', 'a:', 'The answer is:', 'the answer is:']
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
            break
    
    text = text.split('\n')[0].strip()
    
    if ground_truth and text:
        gt_words = len(ground_truth.split())
        text_words = len(text.split())
        
        if gt_words > 0 and text_words > max_ratio * gt_words:
            sentences = re.split(r'[.!?]+', text)
            if sentences and sentences[0].strip():
                text = sentences[0].strip()
    
    return text.strip()

def exact_match(prediction: str, references: List[str]) -> float:
    """Exact match metric"""
    if not references:
        return 0.0
    
    pred = normalize_answer(prediction)
    refs = [normalize_answer(r) for r in references]
    
    return float(any(pred == ref for ref in refs))

def token_f1(prediction: str, references: List[str]) -> float:
    """Token-level F1 score"""
    if not references:
        return 0.0
    
    pred_tokens = normalize_answer(prediction).split()
    ref_token_lists = [normalize_answer(r).split() for r in references]
    
    if not pred_tokens:
        return 0.0
    
    max_f1 = 0.0
    
    for ref_tokens in ref_token_lists:
        if not ref_tokens:
            continue
        
        common = set(pred_tokens) & set(ref_tokens)
        
        if not common:
            continue
        
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(ref_tokens)
        
        f1 = 2 * precision * recall / (precision + recall)
        max_f1 = max(max_f1, f1)
    
    return float(max_f1)

def substring_match(prediction: str, references: List[str]) -> float:
    """Substring match metric"""
    if not references:
        return 0.0
    
    pred = normalize_answer(prediction)
    refs = [normalize_answer(r) for r in references]
    
    return float(any(ref in pred for ref in refs if ref))

print("Normalization and metrics defined")

Statistical Utilities

In [ ]:
def bootstrap_ci(data: List[float], n_bootstrap: int = 1000, confidence: float = 0.95) -> Tuple[float, float, float]:
    """Bootstrap confidence intervals"""
    data = np.array(data)
    
    if data.size == 0:
        return 0.0, 0.0, 0.0
    
    if len(data) == 1:
        val = float(data[0])
        return val, val, val
    
    bootstrap_means = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_means.append(np.mean(sample))
    
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_means, alpha/2 * 100)
    upper = np.percentile(bootstrap_means, (1 - alpha/2) * 100)
    
    return float(np.mean(data)), float(lower), float(upper)

def cohens_d(group1: np.ndarray, group2: np.ndarray) -> float:
    """Cohen's d effect size"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std if pooled_std > 0 else 0.0

def save_json(data: Dict, path: Path):
    """Save JSON with type conversion"""
    def convert(obj):
        if isinstance(obj, (np.integer, np.int64)):
            return int(obj)
        if isinstance(obj, (np.floating, np.float64)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, (np.bool_, bool)):
            return bool(obj)
        if isinstance(obj, dict):
            return {key: convert(value) for key, value in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [convert(item) for item in obj]
        return obj
    
    with open(path, 'w') as f:
        json.dump(convert(data), f, indent=2)

print("Statistical utilities defined")

Load Generated Data

In [ ]:
def load_set_b_data(config_name: str) -> Dict:
    """Load Set B data for a configuration"""
    file_path = INPUT_DIR / f"{config_name}_set_b_complete.json"
    
    if not file_path.exists():
        print(f"WARNING: {file_path.name} not found")
        return None
    
    with open(file_path) as f:
        return json.load(f)

configs = [
    'fp16_base',
    'fp16_instruct',
    'awq_base',
    'awq_instruct',
    'nf4_base',
    'nf4_instruct',
    'gptq_base',
    'gptq_instruct'
]

data = {}
for config in configs:
    loaded = load_set_b_data(config)
    if loaded:
        data[config] = loaded
        print(f"Loaded {config}: {len(loaded['samples'])} samples")

print(f"\nTotal configs loaded: {len(data)}")

Compute Sample Metrics

In [ ]:
def compute_sample_metrics(sample: Dict) -> Dict:
    """Compute metrics for a single sample"""
    ground_truth = sample.get('ground_truth', '')
    ground_truth_variants = sample.get('ground_truth_variants', [])
    if not ground_truth_variants:
        ground_truth_variants = [ground_truth] if ground_truth else []
    
    rag_raw = sample.get('rag_prediction', '')
    rag_extracted = extract_conservative(rag_raw, ground_truth)
    
    em = exact_match(rag_extracted, ground_truth_variants)
    f1 = token_f1(rag_extracted, ground_truth_variants)
    substring = substring_match(rag_extracted, ground_truth_variants)
    
    return {
        'exact_match': em,
        'token_f1': f1,
        'substring_match': substring,
        'target_length': sample.get('target_length_tokens', 0),
        'actual_length': sample.get('actual_length_tokens', 0),
        'answer_position': sample.get('answer_position', 'unknown'),
        'ground_truth': ground_truth,
        'prediction': rag_extracted
    }

print("Sample metrics function defined")

Aggregate by Context Length and Position

In [ ]:
def aggregate_by_length_and_position(samples_metrics: List[Dict]) -> Dict:
    """Aggregate metrics by context length and answer position"""
    
    results = {
        'by_length': {},
        'by_position': {},
        'by_length_and_position': {}
    }
    
    groups_by_length = defaultdict(lambda: defaultdict(list))
    groups_by_position = defaultdict(lambda: defaultdict(list))
    groups_by_both = defaultdict(lambda: defaultdict(list))
    
    for sm in samples_metrics:
        length = sm['target_length']
        position = sm['answer_position']
        
        for metric in ['exact_match', 'token_f1', 'substring_match']:
            groups_by_length[length][metric].append(sm[metric])
            groups_by_position[position][metric].append(sm[metric])
            groups_by_both[(length, position)][metric].append(sm[metric])
    
    for length in CONTEXT_LENGTHS:
        if length not in groups_by_length:
            continue
        
        results['by_length'][length] = {}
        
        for metric in ['exact_match', 'token_f1', 'substring_match']:
            values = groups_by_length[length][metric]
            mean, ci_lower, ci_upper = bootstrap_ci(values, N_BOOTSTRAP, CONFIDENCE_LEVEL)
            
            results['by_length'][length][metric] = {
                'mean': float(mean),
                'std': float(np.std(values)),
                'median': float(np.median(values)),
                'ci_lower': float(ci_lower),
                'ci_upper': float(ci_upper),
                'count': len(values)
            }
    
    for position in ANSWER_POSITIONS:
        if position not in groups_by_position:
            continue
        
        results['by_position'][position] = {}
        
        for metric in ['exact_match', 'token_f1', 'substring_match']:
            values = groups_by_position[position][metric]
            mean, ci_lower, ci_upper = bootstrap_ci(values, N_BOOTSTRAP, CONFIDENCE_LEVEL)
            
            results['by_position'][position][metric] = {
                'mean': float(mean),
                'std': float(np.std(values)),
                'median': float(np.median(values)),
                'ci_lower': float(ci_lower),
                'ci_upper': float(ci_upper),
                'count': len(values)
            }
    
    for (length, position) in groups_by_both.keys():
        key = f"{length}_{position}"
        results['by_length_and_position'][key] = {}
        
        for metric in ['exact_match', 'token_f1', 'substring_match']:
            values = groups_by_both[(length, position)][metric]
            
            results['by_length_and_position'][key][metric] = {
                'mean': float(np.mean(values)),
                'std': float(np.std(values)),
                'count': len(values)
            }
    
    return results

print("Aggregation function defined")

Degradation Analysis

In [ ]:
def analyze_degradation(by_length_results: Dict) -> Dict:
    """Analyze degradation as context length increases"""
    
    degradation = {}
    
    for metric in ['exact_match', 'token_f1', 'substring_match']:
        lengths = []
        means = []
        
        for length in sorted(CONTEXT_LENGTHS):
            if length in by_length_results:
                lengths.append(length)
                means.append(by_length_results[length][metric]['mean'])
        
        if len(lengths) < 2:
            continue
        
        slope, intercept, r_value, p_value, std_err = stats.linregress(lengths, means)
        
        degradation_per_1k = slope * 1000
        
        initial_accuracy = means[0] if means else 0.0
        final_accuracy = means[-1] if means else 0.0
        total_degradation = initial_accuracy - final_accuracy
        relative_degradation_pct = (total_degradation / initial_accuracy * 100) if initial_accuracy > 0 else 0.0
        
        degradation[metric] = {
            'slope': float(slope),
            'degradation_per_1k_tokens': float(degradation_per_1k),
            'r_squared': float(r_value ** 2),
            'p_value': float(p_value),
            'initial_accuracy': float(initial_accuracy),
            'final_accuracy': float(final_accuracy),
            'total_degradation': float(total_degradation),
            'relative_degradation_pct': float(relative_degradation_pct)
        }
    
    return degradation

print("Degradation analysis function defined")

Position Effect Analysis

In [ ]:
def analyze_position_effects(by_position_results: Dict, by_length_and_position: Dict) -> Dict:
    """Analyze how answer position affects accuracy"""
    
    position_effects = {}
    
    for metric in ['exact_match', 'token_f1', 'substring_match']:
        position_means = {}
        
        for position in ANSWER_POSITIONS:
            if position in by_position_results:
                position_means[position] = by_position_results[position][metric]['mean']
        
        if len(position_means) >= 2:
            best_position = max(position_means, key=position_means.get)
            worst_position = min(position_means, key=position_means.get)
            
            position_effects[metric] = {
                'by_position': position_means,
                'best_position': best_position,
                'worst_position': worst_position,
                'range': float(position_means[best_position] - position_means[worst_position])
            }
    
    position_by_length = {}
    
    for length in CONTEXT_LENGTHS:
        position_by_length[length] = {}
        
        for metric in ['exact_match', 'token_f1']:
            position_scores = {}
            
            for position in ANSWER_POSITIONS:
                key = f"{length}_{position}"
                if key in by_length_and_position:
                    position_scores[position] = by_length_and_position[key][metric]['mean']
            
            if position_scores:
                position_by_length[length][metric] = position_scores
    
    return {
        'overall': position_effects,
        'by_length': position_by_length
    }

print("Position effect analysis function defined")

Compute Complete Metrics

In [ ]:
print("Computing comprehensive metrics for all configurations...")
results = {}

for config_name, config_data in data.items():
    print(f"  {config_name}")
    
    samples = config_data['samples']
    samples_metrics = [compute_sample_metrics(sample) for sample in samples]
    
    aggregated = aggregate_by_length_and_position(samples_metrics)
    
    degradation = analyze_degradation(aggregated['by_length'])
    position_effects = analyze_position_effects(aggregated['by_position'], aggregated['by_length_and_position'])
    
    results[config_name] = {
        'by_length': aggregated['by_length'],
        'by_position': aggregated['by_position'],
        'by_length_and_position': aggregated['by_length_and_position'],
        'degradation_analysis': degradation,
        'position_effects': position_effects,
        'num_samples': len(samples)
    }

print("Metrics computed")

Quantization Degradation Comparison

In [ ]:
def compute_quantization_degradation_set_b(results: Dict) -> Dict:
    """Compare quantized models vs FP16 for context robustness"""
    
    variants = ['base', 'instruct']
    quant_methods = ['awq', 'nf4', 'gptq']
    
    degradation_comparison = {}
    
    for variant in variants:
        fp16_config = f'fp16_{variant}'
        
        if fp16_config not in results:
            continue
        
        degradation_comparison[variant] = {}
        
        fp16_degradation = results[fp16_config]['degradation_analysis']
        
        for quant_method in quant_methods:
            quant_config = f'{quant_method}_{variant}'
            
            if quant_config not in results:
                continue
            
            quant_degradation = results[quant_config]['degradation_analysis']
            
            degradation_comparison[variant][quant_method] = {}
            
            for metric in ['exact_match', 'token_f1']:
                fp16_slope = fp16_degradation[metric]['degradation_per_1k_tokens']
                quant_slope = quant_degradation[metric]['degradation_per_1k_tokens']
                
                fp16_total = fp16_degradation[metric]['total_degradation']
                quant_total = quant_degradation[metric]['total_degradation']
                
                degradation_comparison[variant][quant_method][metric] = {
                    'fp16_slope': float(fp16_slope),
                    'quant_slope': float(quant_slope),
                    'slope_difference': float(quant_slope - fp16_slope),
                    'fp16_total_degradation': float(fp16_total),
                    'quant_total_degradation': float(quant_total),
                    'additional_degradation': float(quant_total - fp16_total),
                    'fp16_initial': float(fp16_degradation[metric]['initial_accuracy']),
                    'quant_initial': float(quant_degradation[metric]['initial_accuracy']),
                    'fp16_final': float(fp16_degradation[metric]['final_accuracy']),
                    'quant_final': float(quant_degradation[metric]['final_accuracy'])
                }
    
    return degradation_comparison

print("Computing quantization degradation comparison...")
degradation_comparison = compute_quantization_degradation_set_b(results)
print("Degradation comparison complete")

Cross-Configuration Position Analysis

In [ ]:
def compare_position_effects_across_configs(results: Dict) -> Dict:
    """Compare position effects across all configurations"""
    
    position_comparison = {}
    
    for metric in ['exact_match', 'token_f1']:
        position_comparison[metric] = {}
        
        for config_name, config_results in results.items():
            position_means = {}
            
            for position in ANSWER_POSITIONS:
                if position in config_results['by_position']:
                    position_means[position] = config_results['by_position'][position][metric]['mean']
            
            if position_means:
                best = max(position_means, key=position_means.get)
                worst = min(position_means, key=position_means.get)
                
                position_comparison[metric][config_name] = {
                    'positions': position_means,
                    'best_position': best,
                    'worst_position': worst,
                    'range': float(position_means[best] - position_means[worst])
                }
    
    return position_comparison

print("Computing cross-configuration position analysis...")
position_comparison = compare_position_effects_across_configs(results)
print("Position comparison complete")

Results Summary: Context Length Effects

In [ ]:
print("SET B EVALUATION: CONTEXT LENGTH ROBUSTNESS\n")
print("DEGRADATION BY CONTEXT LENGTH\n")

print(f"{'Config':<20} {'512':<10} {'1024':<10} {'2048':<10} {'4096':<10} {'Total Drop':<12} {'Slope/1k':<12}")

for config_name in configs:
    if config_name not in results:
        continue
    
    by_length = results[config_name]['by_length']
    degradation = results[config_name]['degradation_analysis']
    
    scores = []
    for length in CONTEXT_LENGTHS:
        if length in by_length:
            scores.append(f"{by_length[length]['token_f1']['mean']:.4f}")
        else:
            scores.append("N/A")
    
    total_drop = degradation['token_f1']['total_degradation']
    slope = degradation['token_f1']['degradation_per_1k_tokens']
    
    print(f"{config_name:<20} {scores[0]:<10} {scores[1]:<10} {scores[2]:<10} {scores[3]:<10} {total_drop:<+12.4f} {slope:<+12.6f}")

print("\n\nDEGRADATION STATISTICS (Token F1):\n")
print(f"{'Config':<20} {'R²':<10} {'p-value':<12} {'Initial':<10} {'Final':<10} {'Rel Drop %':<12}")

for config_name in configs:
    if config_name not in results:
        continue
    
    deg = results[config_name]['degradation_analysis']['token_f1']
    
    r_sq = deg['r_squared']
    p_val = deg['p_value']
    initial = deg['initial_accuracy']
    final = deg['final_accuracy']
    rel_drop = deg['relative_degradation_pct']
    
    print(f"{config_name:<20} {r_sq:<10.4f} {p_val:<12.6f} {initial:<10.4f} {final:<10.4f} {rel_drop:<+12.2f}")

Results Summary: Position Effects

In [ ]:
print("\n\nPOSITION EFFECTS (Token F1)\n")
print(f"{'Config':<20} {'Start':<10} {'Middle':<10} {'End':<10} {'Best':<10} {'Range':<10}")

for config_name in configs:
    if config_name not in results:
        continue
    
    by_position = results[config_name]['by_position']
    position_effects = results[config_name]['position_effects']['overall']
    
    scores = []
    for position in ['start', 'middle', 'end']:
        if position in by_position:
            scores.append(f"{by_position[position]['token_f1']['mean']:.4f}")
        else:
            scores.append("N/A")
    
    if 'token_f1' in position_effects:
        best = position_effects['token_f1']['best_position']
        range_val = position_effects['token_f1']['range']
        print(f"{config_name:<20} {scores[0]:<10} {scores[1]:<10} {scores[2]:<10} {best:<10} {range_val:<10.4f}")
    else:
        print(f"{config_name:<20} {scores[0]:<10} {scores[1]:<10} {scores[2]:<10} {'N/A':<10} {'N/A':<10}")

print("\n\nPOSITION EFFECTS BY LENGTH (Token F1):\n")

for length in CONTEXT_LENGTHS:
    print(f"\n{length} tokens:")
    print(f"{'Config':<20} {'Start':<10} {'Middle':<10} {'End':<10}")
    
    for config_name in configs:
        if config_name not in results:
            continue
        
        position_by_length = results[config_name]['position_effects']['by_length']
        
        if length not in position_by_length or 'token_f1' not in position_by_length[length]:
            continue
        
        scores = position_by_length[length]['token_f1']
        
        start = scores.get('start', 0.0)
        middle = scores.get('middle', 0.0)
        end = scores.get('end', 0.0)
        
        print(f"{config_name:<20} {start:<10.4f} {middle:<10.4f} {end:<10.4f}")

Results Summary: Quantization Degradation

In [ ]:
print("\n\nQUANTIZATION DEGRADATION COMPARISON")
print("How quantization affects long-context robustness\n")

for variant in ['base', 'instruct']:
    if variant not in degradation_comparison:
        continue
    
    print(f"\n{variant.upper()} VARIANT:")
    print(f"{'Quant':<10} {'Metric':<15} {'FP16 Slope':<12} {'Quant Slope':<12} {'Diff':<12} {'FP16 Initial':<12} {'Quant Initial':<12}")
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in degradation_comparison[variant]:
            continue
        
        for metric in ['exact_match', 'token_f1']:
            comp = degradation_comparison[variant][quant_method][metric]
            
            fp16_slope = comp['fp16_slope']
            quant_slope = comp['quant_slope']
            diff = comp['slope_difference']
            fp16_init = comp['fp16_initial']
            quant_init = comp['quant_initial']
            
            print(f"{quant_method:<10} {metric:<15} {fp16_slope:<+12.6f} {quant_slope:<+12.6f} {diff:<+12.6f} {fp16_init:<12.4f} {quant_init:<12.4f}")

print("\n\nInterpretation:")
print("  Slope: Change in accuracy per 1000 tokens (negative = degradation)")
print("  Diff: Additional degradation from quantization")
print("  More negative diff = quantization hurts long-context more")

Statistical Tests

In [ ]:
print("\n\nSTATISTICAL TESTS\n")
print("Testing if degradation slopes differ significantly across quantization methods\n")

for variant in ['base', 'instruct']:
    fp16_config = f'fp16_{variant}'
    
    if fp16_config not in results:
        continue
    
    print(f"\n{variant.upper()} VARIANT (Token F1):")
    print(f"{'Comparison':<30} {'t-stat':<12} {'p-value':<12} {'Cohen d':<12} {'Sig':<5}")
    
    fp16_samples = []
    for sample in data[fp16_config]['samples']:
        rag_pred = sample.get('rag_prediction', '')
        ground_truth = sample.get('ground_truth', '')
        ground_truth_variants = sample.get('ground_truth_variants', [])
        if not ground_truth_variants:
            ground_truth_variants = [ground_truth] if ground_truth else []
        
        extracted = extract_conservative(rag_pred, ground_truth)
        score = token_f1(extracted, ground_truth_variants)
        fp16_samples.append(score)
    
    fp16_samples = np.array(fp16_samples)
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        quant_config = f'{quant_method}_{variant}'
        
        if quant_config not in data:
            continue
        
        quant_samples = []
        for sample in data[quant_config]['samples']:
            rag_pred = sample.get('rag_prediction', '')
            ground_truth = sample.get('ground_truth', '')
            ground_truth_variants = sample.get('ground_truth_variants', [])
            if not ground_truth_variants:
                ground_truth_variants = [ground_truth] if ground_truth else []
            
            extracted = extract_conservative(rag_pred, ground_truth)
            score = token_f1(extracted, ground_truth_variants)
            quant_samples.append(score)
        
        quant_samples = np.array(quant_samples)
        
        t_stat, p_value = stats.ttest_ind(fp16_samples, quant_samples)
        effect_size = cohens_d(fp16_samples, quant_samples)
        
        sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
        
        comparison_name = f"FP16 vs {quant_method.upper()}"
        print(f"{comparison_name:<30} {t_stat:<+12.4f} {p_value:<12.6f} {effect_size:<+12.4f} {sig:<5}")

Save Complete Results

In [ ]:
final_results = {
    'evaluation': 'Set B - Context Length Robustness',
    'configs': {
        config_name: {
            'by_length': config_results['by_length'],
            'by_position': config_results['by_position'],
            'by_length_and_position': config_results['by_length_and_position'],
            'degradation_analysis': config_results['degradation_analysis'],
            'position_effects': config_results['position_effects'],
            'num_samples': config_results['num_samples']
        }
        for config_name, config_results in results.items()
    },
    'quantization_degradation': degradation_comparison,
    'position_comparison': position_comparison,
    'metadata': {
        'n_bootstrap': N_BOOTSTRAP,
        'confidence_level': CONFIDENCE_LEVEL,
        'alpha': ALPHA,
        'random_seed': RANDOM_SEED,
        'context_lengths': CONTEXT_LENGTHS,
        'answer_positions': ANSWER_POSITIONS,
        'metrics': ['exact_match', 'token_f1', 'substring_match'],
        'analyses': ['degradation_curves', 'position_effects', 'quantization_comparison']
    }
}

output_path = OUTPUT_DIR / 'set_b_complete_results.json'
save_json(final_results, output_path)
print(f"\nComplete results saved to: {output_path}")

degradation_summary = []

for config_name in configs:
    if config_name not in results:
        continue
    
    deg = results[config_name]['degradation_analysis']
    
    row = {
        'config': config_name
    }
    
    for metric in ['exact_match', 'token_f1']:
        row[f'{metric}_slope'] = deg[metric]['degradation_per_1k_tokens']
        row[f'{metric}_r_squared'] = deg[metric]['r_squared']
        row[f'{metric}_initial'] = deg[metric]['initial_accuracy']
        row[f'{metric}_final'] = deg[metric]['final_accuracy']
        row[f'{metric}_total_drop'] = deg[metric]['total_degradation']
        row[f'{metric}_rel_drop_pct'] = deg[metric]['relative_degradation_pct']
    
    degradation_summary.append(row)

degradation_df = pd.DataFrame(degradation_summary)
csv_path = OUTPUT_DIR / 'degradation_summary.csv'
degradation_df.to_csv(csv_path, index=False)
print(f"Degradation summary CSV: {csv_path}")

length_breakdown = []

for config_name in configs:
    if config_name not in results:
        continue
    
    by_length = results[config_name]['by_length']
    
    for length in CONTEXT_LENGTHS:
        if length not in by_length:
            continue
        
        row = {
            'config': config_name,
            'context_length': length
        }
        
        for metric in ['exact_match', 'token_f1', 'substring_match']:
            row[f'{metric}_mean'] = by_length[length][metric]['mean']
            row[f'{metric}_std'] = by_length[length][metric]['std']
            row[f'{metric}_ci_lower'] = by_length[length][metric]['ci_lower']
            row[f'{metric}_ci_upper'] = by_length[length][metric]['ci_upper']
        
        length_breakdown.append(row)

length_df = pd.DataFrame(length_breakdown)
csv_path = OUTPUT_DIR / 'by_length_breakdown.csv'
length_df.to_csv(csv_path, index=False)
print(f"By-length breakdown CSV: {csv_path}")

position_breakdown = []

for config_name in configs:
    if config_name not in results:
        continue
    
    by_position = results[config_name]['by_position']
    
    for position in ANSWER_POSITIONS:
        if position not in by_position:
            continue
        
        row = {
            'config': config_name,
            'position': position
        }
        
        for metric in ['exact_match', 'token_f1', 'substring_match']:
            row[f'{metric}_mean'] = by_position[position][metric]['mean']
            row[f'{metric}_std'] = by_position[position][metric]['std']
        
        position_breakdown.append(row)

position_df = pd.DataFrame(position_breakdown)
csv_path = OUTPUT_DIR / 'by_position_breakdown.csv'
position_df.to_csv(csv_path, index=False)
print(f"By-position breakdown CSV: {csv_path}")

quant_degradation_summary = []

for variant in ['base', 'instruct']:
    if variant not in degradation_comparison:
        continue
    
    for quant_method in ['awq', 'nf4', 'gptq']:
        if quant_method not in degradation_comparison[variant]:
            continue
        
        row = {
            'variant': variant,
            'quant_method': quant_method
        }
        
        for metric in ['exact_match', 'token_f1']:
            comp = degradation_comparison[variant][quant_method][metric]
            row[f'{metric}_fp16_slope'] = comp['fp16_slope']
            row[f'{metric}_quant_slope'] = comp['quant_slope']
            row[f'{metric}_slope_diff'] = comp['slope_difference']
            row[f'{metric}_additional_degradation'] = comp['additional_degradation']
        
        quant_degradation_summary.append(row)

quant_deg_df = pd.DataFrame(quant_degradation_summary)
csv_path = OUTPUT_DIR / 'quantization_degradation_summary.csv'
quant_deg_df.to_csv(csv_path, index=False)
print(f"Quantization degradation summary CSV: {csv_path}")

print("\nEVALUATION COMPLETE")
print("Set B includes: Context Length Robustness, Position Effects, and Quantization Impact on Long Context")